# UCI Online Retail — PM Quickstart

Load the committed sample, profile it, and show three PM-relevant views:
1. RFM segmentation
2. Top countries & products
3. Returns rate

**Dataset**: [`samples/uci-online-retail-sample.csv`](../samples/uci-online-retail-sample.csv) — first 500 rows of UCI Online Retail (CC BY 4.0).

For the full 541K-row dataset, run `python scripts/fetch_samples.py uci` first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

df = pd.read_csv('../samples/uci-online-retail-sample.csv', parse_dates=['InvoiceDate'])
print('Shape:', df.shape)
df.head()

## 1. Profile the columns

In [ ]:
print('Date range:', df['InvoiceDate'].min(), '→', df['InvoiceDate'].max())
print('Unique invoices:', df['InvoiceNo'].nunique())
print('Unique customers:', df['CustomerID'].nunique())
print('Unique countries:', df['Country'].nunique())
print('Nulls per column:')
print(df.isna().sum())

## 2. Top countries and products

In [ ]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.groupby('Country')['Revenue'].sum().nlargest(10).plot.barh(ax=axes[0], title='Top 10 countries by revenue')
df.groupby('Description')['Quantity'].sum().nlargest(10).plot.barh(ax=axes[1], title='Top 10 products by units sold')
axes[0].invert_yaxis(); axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Returns rate

Cancellations have invoice numbers starting with `C`.

In [ ]:
df['IsCancellation'] = df['InvoiceNo'].astype(str).str.startswith('C')

returns = df[df['IsCancellation']]
sales   = df[~df['IsCancellation']]

print(f'Sales rows:       {len(sales):,}')
print(f'Cancellation rows: {len(returns):,}')
print(f'Returns rate:     {len(returns) / len(df):.2%}')

if not returns.empty:
    print('\nTop returned products:')
    print(returns.groupby('Description')['Quantity'].sum().abs().nlargest(5))

## 4. RFM segmentation

Compute Recency / Frequency / Monetary per customer. With only 500 rows this is illustrative; rerun on the full file for real cohorts.

In [ ]:
clean = df[(~df['IsCancellation']) & (df['CustomerID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot = clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum'),
).reset_index()

for col in ['Recency', 'Frequency', 'Monetary']:
    rfm[f'{col[0]}_quintile'] = pd.qcut(rfm[col].rank(method='first'), 5, labels=[5,4,3,2,1] if col == 'Recency' else [1,2,3,4,5])

rfm['RFM_Score'] = rfm['R_quintile'].astype(int) + rfm['F_quintile'].astype(int) + rfm['M_quintile'].astype(int)
rfm.sort_values('RFM_Score', ascending=False).head(10)

## What to try next
- Run on the full UCI file (541K rows) — `python scripts/fetch_samples.py uci`.
- Add `IsWholesale` flag (Quantity > 10 or Revenue > £200).
- Build cohort retention curves grouped by first-purchase month.
- Read [`scorecards/01-customer-health-rfm.md`](../scorecards/01-customer-health-rfm.md) for the metrics framework.